# **CLC Face Detection on a Microcontroller**

In this project we are designing a model that is able to detect a face when deployed on a microcontroller. It is important to note that recognitino is different than recognition. Essentialy, our model says "Yup, that's a face!" instead of explicity identifying who the face belongs to.
________________

## 1. Introduction and objective

**Project Goal:** Train a neural network to detect faces in frame on a camera connected to an Arduino microcontroller. The system must distinguish between what is a face and what is not. Then, the model will be deployed to run efficiently on resource-constrained hardware.

**Objectives:**
- Detect if a face is present in-frame on a camera.
- Detect if a face is **NOT** present.
- Obtain an initial accuracy score, then successfully improve that accuracy score to at least 90%.
- Improve data extraction and processing techniques.
- Model optimization.
- Deployment on a microcontroller
- Achieve > 80% accuracy on the first attempt and > 90% on the second attempt.
_______________

## 2. Data Acquisition

We have a file in our project called
```python
DataImporter.py
```
which extracts faces from the LFW dataset and random images such as animals and vehicles from the CIFAR-10 dataset. These datasets belong to the sklearn library. In this file, images are pulled in the size of 48x48 pixels. This means the resolution is lowered (not screteched, compressed, or cropped).

Example:

In [ ]:
from sklearn.datasets import fetch_lfw_people

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = os.path.join(os.path.dirname(__file__), "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")

# Image size for the model input (grayscale)
IMG_SIZE = 48


def resize_images(images, target_size):
    """Resize a batch of 2D images to target_size x target_size using nearest-neighbor."""
    n = len(images)
    resized = np.zeros((n, target_size, target_size), dtype=images.dtype)
    for i in range(n):
        h, w = images[i].shape[:2]
        row_idx = np.clip(np.floor(np.arange(target_size) * h / target_size).astype(int), 0, h - 1)
        col_idx = np.clip(np.floor(np.arange(target_size) * w / target_size).astype(int), 0, w - 1)
        resized[i] = images[i][np.ix_(row_idx, col_idx)]
    return resized


# Split each class: 80% train, 10% val, 10% test
    def split_data(images, labels):
        n = len(images)
        n_val = n // 10
        n_test = n // 10
        n_train = n - n_val - n_test

        splits = {
            "train": (images[:n_train], labels[:n_train]),
            "val": (images[n_train:n_train + n_val], labels[n_train:n_train + n_val]),
            "test": (images[n_train + n_val:], labels[n_train + n_val:]),
        }
        return splits

    face_splits = split_data(face_images, face_labels)
    nonface_splits = split_data(nonface_images, nonface_labels)


________________________________________

## 3. Data Preprocessing

We also have a file called 
```
DataPreprocessor.py
```
which takes the 48x48 images and they are converted to grayscale. They are also augmented (flipped or rotated) and then saved to .npy files for training.

In [ ]:
import os
import json
import random
import numpy as np

from DataImporter import import_data, IMG_SIZE

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = os.path.join(os.path.dirname(__file__), "data", "processed")

LABEL_NAMES = ["non_face", "face"]
LABEL_TO_INDEX = {name: i for i, name in enumerate(LABEL_NAMES)}


def resize_image(image, target_size=IMG_SIZE):
    """Resize an image to target_size x target_size using bilinear interpolation.

    Uses numpy-only interpolation to avoid extra dependencies.
    """
    h, w = image.shape[:2]
    if h == target_size and w == target_size:
        return image

    # Simple bilinear resize using numpy
    row_ratio = h / target_size
    col_ratio = w / target_size

    row_idx = np.floor(np.arange(target_size) * row_ratio).astype(int)
    col_idx = np.floor(np.arange(target_size) * col_ratio).astype(int)

    row_idx = np.clip(row_idx, 0, h - 1)
    col_idx = np.clip(col_idx, 0, w - 1)

    resized = image[np.ix_(row_idx, col_idx)]
    return resized


def augment_flip(image):
    """Randomly flip image horizontally."""
    if random.random() < 0.5:
        return np.fliplr(image)
    return image


def augment_brightness(image, max_delta=0.2):
    """Randomly adjust brightness."""
    delta = random.uniform(-max_delta, max_delta)
    return np.clip(image + delta, 0.0, 1.0)


def augment_noise(image, noise_std=0.02):
    """Add random gaussian noise."""
    noise = np.random.normal(0, noise_std, image.shape).astype(np.float32)
    return np.clip(image + noise, 0.0, 1.0)


def augment_crop_and_resize(image, min_crop=0.85):
    """Random crop and resize back to original size."""
    h, w = image.shape
    crop_frac = random.uniform(min_crop, 1.0)
    new_h = int(h * crop_frac)
    new_w = int(w * crop_frac)

    top = random.randint(0, h - new_h)
    left = random.randint(0, w - new_w)

    cropped = image[top:top + new_h, left:left + new_w]
    return resize_image(cropped, h)


def preprocess_data():
    """Run the full preprocessing pipeline."""
    print("Importing data...")
    data = import_data()

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for split in ["train", "val", "test"]:
        images = data[split]["images"]
        labels = data[split]["labels"]
        n_samples = len(images)
        print(f"\nProcessing {split} set ({n_samples} samples)...")

        processed = []

        for i in range(n_samples):
            if (i + 1) % 500 == 0 or i == n_samples - 1:
                print(f"  {i + 1}/{n_samples}")

            img = images[i].astype(np.float32)

            # Normalize to [0, 1] if not already
            if img.max() > 1.0:
                img = img / 255.0

            # Resize to target size
            img = resize_image(img, IMG_SIZE)

            # Apply augmentations to training data only
            if split == "train":
                img = augment_flip(img)
                if random.random() < 0.5:
                    img = augment_brightness(img)
                if random.random() < 0.3:
                    img = augment_noise(img)
                if random.random() < 0.3:
                    img = augment_crop_and_resize(img)

            processed.append(img)

        X = np.array(processed, dtype=np.float32)

        # Add channel dimension: (N, 48, 48) -> (N, 48, 48, 1)
        X = X[..., np.newaxis]

        # Compute normalization stats on training set
        if split == "train":
            mean = X.mean()
            std = X.std()
            if std == 0:
                std = 1.0
            np.save(os.path.join(OUTPUT_DIR, "mean.npy"), np.array([mean]))
            np.save(os.path.join(OUTPUT_DIR, "std.npy"), np.array([std]))
        else:
            mean = np.load(os.path.join(OUTPUT_DIR, "mean.npy"))[0]
            std = np.load(os.path.join(OUTPUT_DIR, "std.npy"))[0]

        X = (X - mean) / std

        y = labels.astype(np.int32)

        np.save(os.path.join(OUTPUT_DIR, f"X_{split}.npy"), X)
        np.save(os.path.join(OUTPUT_DIR, f"y_{split}.npy"), y)
        print(f"  Saved X_{split}.npy: {X.shape}, y_{split}.npy: {y.shape}")

    # Save label mapping
    label_map_path = os.path.join(OUTPUT_DIR, "label_map.json")
    with open(label_map_path, "w") as f:
        json.dump(LABEL_TO_INDEX, f, indent=2)
    print(f"\nLabel mapping saved to {label_map_path}")
    print("Done!")


if __name__ == "__main__":
    preprocess_data()

__________________

## 4. Model Architecture

- Input: 48x48x1 grayscale images
- Conv2D(8 filters, 3x3, RelU) -> 48x48x8
- Conv2D(16 filters, 3x3, RelU) -> 24x24x16
- Conv2D( 32 filters, 3x3, ReLU) -> 12x12x32
- GlobalAveragePooling2D()
- Droput(0.3)

In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = os.path.join(os.path.dirname(__file__), "data", "processed")
ARDUINO_DIR = os.path.join(os.path.dirname(__file__), "face_detector_arduino")

NUM_CLASSES = 2
IMG_SIZE = 48

def build_model(input_shape):
    """Build a tiny 2D CNN for face detection (50% smaller model).

    Architecture designed for Arduino deployment:
        Conv2D(8) -> MaxPool -> Conv2D(16) -> MaxPool -> Conv2D(32) -> GAP -> Dense(2)

    This is 50% smaller than the original, reducing model size and memory usage.
    """
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),

        # 48x48x1 -> 24x24x8
        tf.keras.layers.Conv2D(8, (3, 3), padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),

        # 24x24x8 -> 12x12x16
        tf.keras.layers.Conv2D(16, (3, 3), padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),

        # 12x12x16 -> 6x6x32
        tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),

        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
    ])

    def loss_with_label_smoothing(y_true, y_pred):
        smoothing = 0.1
        y_true_onehot = tf.one_hot(tf.cast(y_true, tf.int32), NUM_CLASSES)
        y_true_smooth = y_true_onehot * (1.0 - smoothing) + smoothing / NUM_CLASSES
        return tf.keras.losses.categorical_crossentropy(y_true_smooth, y_pred)

    model.compile(
        optimizer='adam',
        loss=loss_with_label_smoothing,
        metrics=['accuracy'],
    )

    model.summary()
    return model

_____________________________
## 5. Model training
- Forward pass: batches of 64 images are fed through the CNN                                                                        
- Loss computed using the label smoothing cross-entropy loss defined in build_model()
- Backpropagation: Adam optimizer updates weights to minimize loss
- Validation after each epoch, performance is measured on the held-out validation set (no weight updates)

Two callbacks monitor val_loss and adjust training automatically:

  EarlyStopping (patience=15):
  - Watches validation loss each epoch
  - If it doesn't improve for 15 consecutive epochs, training stops
  - restore_best_weights=True rewinds weights to the epoch with the lowest val_loss, not the last epoch

  ReduceLROnPlateau (patience=8, factor=0.5):
  - If val_loss stagnates for 8 epochs, the learning rate is cut in half
  - This lets the optimizer take smaller, more precise steps to escape plateaus
  - Floor is min_lr=1e-6 to prevent the LR from shrinking to zero

In [ ]:
def train_model(model, X_train, y_train, X_val, y_val):
    """Train the model with early stopping and learning rate reduction."""
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=15,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=8,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=200,
        batch_size=64,
        callbacks=callbacks,
        verbose=1,
    )

    return history


def evaluate_model(model, X_test, y_test, index_to_label):
    """Evaluate the model and print per-class metrics."""
    print("\n" + "=" * 60)
    print("TEST SET EVALUATION")
    print("=" * 60)

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"\nTest Loss:     {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")

    if test_acc < 0.90:
        print("WARNING: Accuracy is below 90% target!")
    else:
        print("PASS: Accuracy meets >90% target.")

    y_pred = model.predict(X_test, verbose=0)
    y_pred_classes = np.argmax(y_pred, axis=1)

    label_names = [index_to_label[i] for i in range(len(index_to_label))]
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_classes, target_names=label_names))

    print("Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred_classes)
    header = "          " + "  ".join(f"{name[:8]:>8}" for name in label_names)
    print(header)
    for i, row in enumerate(cm):
        row_str = f"{label_names[i]:>9} " + "  ".join(f"{val:>8}" for val in row)
        print(row_str)

    return test_acc


def main():
    print("=" * 60)
    print("Face Detection - CNN Training")
    print("=" * 60)

    gpu_devices = tf.config.list_physical_devices('GPU')
    if gpu_devices:
        for gpu in gpu_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Training on GPU: {gpu_devices[0].name}")
    else:
        print("Training on CPU (no GPU detected)")

    X_train, y_train, X_val, y_val, X_test, y_test, label_map, index_to_label = load_data()

    model = build_model(input_shape=(IMG_SIZE, IMG_SIZE, 1))

    print("\n" + "=" * 60)
    print("TRAINING")
    print("=" * 60)
    train_model(model, X_train, y_train, X_val, y_val)

    test_acc = evaluate_model(model, X_test, y_test, index_to_label)

    print("\n" + "=" * 60)
    print("TFLITE CONVERSION")
    print("=" * 60)
    tflite_model = convert_to_tflite(model)
    generate_c_header(tflite_model, label_map, index_to_label)

    keras_path = os.path.join(os.path.dirname(__file__), "face_model.keras")
    model.save(keras_path)
    print(f"Saved Keras model: {keras_path}")

    print("\n" + "=" * 60)
    print(f"DONE - Test accuracy: {test_acc:.4f}")
    if test_acc >= 0.90:
        print("Target >90% accuracy: ACHIEVED")
    else:
        print("Target >90% accuracy: NOT MET - consider more epochs or data")
    print("=" * 60)


if __name__ == "__main__":
    main()

________________________
## 6. Model Testing

We have a simple function that does model testing from a python perspective:

In [ ]:
def evaluate_model(model, X_test, y_test, index_to_label):
    """Evaluate the model and print per-class metrics."""
    print("\n" + "=" * 60)
    print("TEST SET EVALUATION")
    print("=" * 60)

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"\nTest Loss:     {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")

    if test_acc < 0.90:
        print("WARNING: Accuracy is below 90% target!")
    else:
        print("PASS: Accuracy meets >90% target.")

    y_pred = model.predict(X_test, verbose=0)
    y_pred_classes = np.argmax(y_pred, axis=1)

    label_names = [index_to_label[i] for i in range(len(index_to_label))]
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_classes, target_names=label_names))

    print("Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred_classes)
    header = "          " + "  ".join(f"{name[:8]:>8}" for name in label_names)
    print(header)
    for i, row in enumerate(cm):
        row_str = f"{label_names[i]:>9} " + "  ".join(f"{val:>8}" for val in row)
        print(row_str)

    return test_acc

__________________________
## 7. Conversion to TFLite

The training file also converts to TFLite and generates a C header file to push to the arduino

In [ ]:
def convert_to_tflite(model):
    """Convert Keras model to float32 TFLite format."""
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()

    tflite_path = os.path.join(os.path.dirname(__file__), "face_model.tflite")
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)

    print(f"\nSaved TFLite model: {tflite_path} ({len(tflite_model)} bytes)")

    interpreter = tf.lite.Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    print(f"TFLite input shape:  {input_details[0]['shape']}, dtype: {input_details[0]['dtype']}")
    print(f"TFLite output shape: {output_details[0]['shape']}, dtype: {output_details[0]['dtype']}")
    print("TFLite model verification successful!")

    return tflite_model


def generate_c_header(tflite_model, label_map, index_to_label):
    """Generate C header file for Arduino deployment."""
    os.makedirs(ARDUINO_DIR, exist_ok=True)
    header_path = os.path.join(ARDUINO_DIR, "face_model_data.h")

    mean = np.load(os.path.join(DATA_DIR, "mean.npy")).flatten()
    std = np.load(os.path.join(DATA_DIR, "std.npy")).flatten()

    with open(header_path, 'w') as f:
        f.write("// Face Detection TFLite model and configuration\n")
        f.write("// Auto-generated by train_cnn_model.py - do not edit\n")
        f.write("#ifndef FACE_MODEL_DATA_H\n")
        f.write("#define FACE_MODEL_DATA_H\n\n")

        f.write(f"// Model size: {len(tflite_model)} bytes\n")
        f.write("alignas(8) const unsigned char face_model_tflite[] = {\n")
        for i in range(0, len(tflite_model), 12):
            row = tflite_model[i:i+12]
            hex_vals = ', '.join(f'0x{b:02x}' for b in row)
            if i + 12 < len(tflite_model):
                f.write(f"  {hex_vals},\n")
            else:
                f.write(f"  {hex_vals}\n")
        f.write("};\n\n")
        f.write(f"const unsigned int face_model_tflite_len = {len(tflite_model)};\n\n")

        f.write(f"const int kImageSize = {IMG_SIZE};\n")
        f.write(f"const int kImageChannels = 1;  // grayscale\n\n")

        num_classes = len(label_map)
        f.write(f"const int kNumClasses = {num_classes};\n\n")

        f.write("const char* const kLabelNames[] = {\n")
        for i in range(num_classes):
            name = index_to_label[i]
            comma = "," if i < num_classes - 1 else ""
            f.write(f'  "{name}"{comma}\n')
        f.write("};\n\n")

        f.write("// Pixel normalization: (pixel / 255.0 - mean) / std\n")
        f.write(f"const float kPixelMean = {mean[0]:.6f}f;\n")
        f.write(f"const float kPixelStd = {std[0]:.6f}f;\n\n")

        f.write("#endif\n")

    print(f"Saved C header: {header_path}")